# SAM3 Diagnostic Notebook

This notebook exposes **image inference**, **video inference**, and **visualization** as independent cells.

**Purpose**: Determine whether empty masks are caused by the bfloat16 / CPU-offload modifications or by the original code.

**How to use**:
1. Run Cell 1 (Clone & Install) → Cell 1b (Weights) → Cell 2 (Config)
2. Toggle `USE_BFLOAT16` / `OFFLOAD_VIDEO_CPU` in Cell 2
3. Run the inference + visualization cells you need
4. Compare results between flag combinations

**Runtime**: Go to `Runtime → Change runtime type` and select **T4 GPU** (free) or better.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 1 — Clone Repo & Install
# ═══════════════════════════════════════════════════════════════════════

import os, sys

# ── Colab detection (robust) ──────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"Running in Colab: {IN_COLAB}")

# ── Clone & install ───────────────────────────────────────────────────
# ⚠️  Replace with YOUR GitHub username and repo name
GITHUB_USERNAME = "YOUR_USERNAME"
REPO_NAME       = "YOUR_REPO_NAME"
REPO_URL        = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR        = f"/content/{REPO_NAME}"

# If the repo is PRIVATE, use a token:
# from google.colab import userdata
# token = userdata.get('GITHUB_TOKEN')   # set in Colab Secrets (🔑 icon)
# REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        print(f"Cloning {REPO_URL} ...")
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        print(f"Repo already cloned at {REPO_DIR}, pulling latest...")
        os.system(f"git -C {REPO_DIR} pull")

    os.chdir(REPO_DIR)
    print(f"Working directory: {os.getcwd()}")

    # Install SAM3 package in editable mode (reads pyproject.toml)
    print("\nInstalling SAM3 package...")
    os.system("pip install -e . -q")

    # Install notebook dependencies
    print("Installing notebook dependencies...")
    os.system("pip install opencv-python matplotlib pillow tqdm -q")

else:
    # Local: ensure repo root is on sys.path
    repo_root = os.path.abspath(os.path.join(os.getcwd()))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)
    print(f"Local mode. sys.path includes: {repo_root}")

# ── Verify PyTorch & CUDA ─────────────────────────────────────────────
import torch
print(f"\nPyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB")
    os.system("nvidia-smi")
else:
    print("⚠ WARNING: No GPU detected. Inference will be very slow on CPU.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 1b — Download Model Checkpoint Weights
# ═══════════════════════════════════════════════════════════════════════
# Run this ONCE. The checkpoint is saved to ./checkpoints/ inside your repo.
# If you already have the weights on Google Drive, see the Drive mount option below.

import os

os.makedirs("checkpoints", exist_ok=True)

# ── Option A: Download directly from Meta (choose one size) ───────────
#
# ⚠️  Verify the exact checkpoint filename your model_builder.py expects.
#    Common names: sam3_hiera_large.pt  /  sam3_hiera_small.pt  /  sam3_hiera_base_plus.pt
#
CHECKPOINT_NAME = "sam3_hiera_large.pt"   # ← change if needed
CHECKPOINT_URL  = f"https://dl.fbaipublicfiles.com/segment_anything_3/{CHECKPOINT_NAME}"
CHECKPOINT_PATH = f"checkpoints/{CHECKPOINT_NAME}"

if not os.path.exists(CHECKPOINT_PATH):
    print(f"Downloading {CHECKPOINT_NAME} ...")
    ret = os.system(f"wget -q --show-progress {CHECKPOINT_URL} -O {CHECKPOINT_PATH}")
    if ret == 0 and os.path.exists(CHECKPOINT_PATH):
        size_gb = os.path.getsize(CHECKPOINT_PATH) / 1024**3
        print(f"✅ Downloaded: {CHECKPOINT_PATH}  ({size_gb:.2f} GiB)")
    else:
        print("❌ Download failed. Check the URL or use Option B below.")
else:
    size_gb = os.path.getsize(CHECKPOINT_PATH) / 1024**3
    print(f"✅ Checkpoint already exists: {CHECKPOINT_PATH}  ({size_gb:.2f} GiB)")

# ── Option B: Mount Google Drive to avoid re-downloading each session ─
# Uncomment and run if you've saved the .pt file to Google Drive.
#
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_CKPT = '/content/drive/MyDrive/sam3_checkpoints/sam3_hiera_large.pt'
# if not os.path.exists(CHECKPOINT_PATH):
#     os.system(f"cp {DRIVE_CKPT} {CHECKPOINT_PATH}")
#     print(f"✅ Copied checkpoint from Drive: {CHECKPOINT_PATH}")

print(f"\nAll files in checkpoints/:")
for f in os.listdir("checkpoints"):
    sz = os.path.getsize(f"checkpoints/{f}") / 1024**3
    print(f"  {f}  ({sz:.2f} GiB)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 1c — Verify Repo Structure & Custom Files
# ═══════════════════════════════════════════════════════════════════════
# Sanity check that all YOUR custom scripts and the sam3 package are
# accessible before running inference.

import os

print(f"Current directory: {os.getcwd()}")
print()

files_to_check = [
    ("run_sam3_inference.py",    "Your inference runner"),
    ("debug_masks.py",           "Your debug script"),
    ("visualize_masks.py",       "Your visualizer"),
    ("test_mask_type.py",        "Your mask type tester"),
    ("pyproject.toml",           "Package config"),
    ("sam3/__init__.py",         "SAM3 package"),
    ("sam3/model_builder.py",    "Model builder"),
    (f"checkpoints/{CHECKPOINT_NAME}", "Model weights"),
]

all_ok = True
for fpath, label in files_to_check:
    exists = os.path.exists(fpath)
    status = "✅" if exists else "❌ MISSING"
    if not exists:
        all_ok = False
    print(f"  {status}  {fpath:<45} ({label})")

print()

# Verify sam3 is importable as a package
try:
    import sam3
    print(f"✅ sam3 package imported from: {sam3.__file__}")
    print(f"   sam3 version: {sam3.__version__}")
except Exception as e:
    print(f"❌ Failed to import sam3: {e}")
    all_ok = False

print()
if all_ok:
    print("✅ All checks passed — safe to proceed to Cell 2.")
else:
    print("⚠ Some files are missing. Check the clone step in Cell 1 and re-run.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 2 — Configuration Flags
# ═══════════════════════════════════════════════════════════════════════
#
# Toggle these to test different modes:
#   - Both False  → original unmodified SAM3 behavior
#   - Both True   → VRAM-optimized path (bfloat16 + CPU offload)
#   - Mix         → isolate which flag causes issues
#
# ───────────────────────────────────────────────────────────────────────

USE_BFLOAT16       = False   # Cast model weights to bfloat16 (halves VRAM usage)
OFFLOAD_VIDEO_CPU  = False   # Keep video frames on CPU until needed
DEVICE             = "cuda" if __import__('torch').cuda.is_available() else "cpu"
CONFIDENCE_THRESH  = 0.5     # Detection confidence threshold

# ─── Input paths ──────────────────────────────────────────────────────
IMAGE_PATH          = "inputs/images/sample_image.jpg"
VIDEO_PATH          = "inputs/videos/3-dogs.mp4"
TEXT_PROMPT_IMAGE   = "shoe"
TEXT_PROMPT_VIDEO   = "dogs"

# ─── Output dirs ──────────────────────────────────────────────────────
import os
os.makedirs("outputs/overlay",    exist_ok=True)
os.makedirs("outputs/diagnostic", exist_ok=True)

# ─── Quick input file check ───────────────────────────────────────────
for path in [IMAGE_PATH, VIDEO_PATH]:
    status = "✅" if os.path.exists(path) else "⚠ NOT FOUND (upload or adjust path above)"
    print(f"  {status}  {path}")

print("\n" + "=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"  USE_BFLOAT16      = {USE_BFLOAT16}")
print(f"  OFFLOAD_VIDEO_CPU = {OFFLOAD_VIDEO_CPU}")
print(f"  DEVICE            = {DEVICE}")
print(f"  CONFIDENCE_THRESH = {CONFIDENCE_THRESH}")
print(f"  IMAGE_PATH        = {IMAGE_PATH}")
print(f"  VIDEO_PATH        = {VIDEO_PATH}")
print(f"  TEXT_PROMPT_IMAGE = {TEXT_PROMPT_IMAGE}")
print(f"  TEXT_PROMPT_VIDEO = {TEXT_PROMPT_VIDEO}")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 3 — Image Model: Build & Inference
# ═══════════════════════════════════════════════════════════════════════

import torch
import numpy as np
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("[Image] Building model...")
image_model = build_sam3_image_model()

if USE_BFLOAT16:
    print("[Image] Casting model to bfloat16...")
    image_model.to(dtype=torch.bfloat16)

image_model.to(device=DEVICE)
print(f"[Image] Model dtype  : {next(image_model.parameters()).dtype}")
print(f"[Image] Model device : {next(image_model.parameters()).device}")

processor = Sam3Processor(image_model, confidence_threshold=CONFIDENCE_THRESH, device=DEVICE)

print(f"\n[Image] Loading image: {IMAGE_PATH}")
pil_image = Image.open(IMAGE_PATH)
print(f"[Image] Image size   : {pil_image.size}")

inference_state = processor.set_image(pil_image)

print(f"[Image] Running inference with prompt: '{TEXT_PROMPT_IMAGE}'...")
inference_state = processor.set_text_prompt(state=inference_state, prompt=TEXT_PROMPT_IMAGE)

img_masks  = inference_state["masks"]
img_boxes  = inference_state["boxes"]
img_scores = inference_state["scores"]

print("\n" + "=" * 60)
print("IMAGE INFERENCE RESULTS")
print("=" * 60)
print(f"  Masks found    : {len(img_masks)}")
if len(img_masks) > 0:
    print(f"  Masks shape    : {img_masks.shape}")
    print(f"  Masks dtype    : {img_masks.dtype}")
    for i in range(min(len(img_masks), 5)):
        m = img_masks[i]
        if hasattr(m, 'cpu'):
            m = m.cpu()
        nz = m.sum().item()
        print(f"  Mask {i}: score={img_scores[i].item():.4f}  non-zero pixels={nz}  shape={m.shape}")
    if all((img_masks[i].cpu().sum().item() == 0) for i in range(len(img_masks))):
        print("\n  ⚠  WARNING: ALL masks are entirely zero!")
    else:
        print("\n  ✓  At least one mask has non-zero pixels.")
else:
    print("  ⚠  WARNING: No masks detected at all!")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 4 — Video Model: Build & Inference
# ═══════════════════════════════════════════════════════════════════════

import torch
import numpy as np
from sam3.model_builder import build_sam3_video_predictor

print("[Video] Building predictor...")
video_predictor = build_sam3_video_predictor()

if USE_BFLOAT16:
    print("[Video] Casting model to bfloat16...")
    video_predictor.model.to(dtype=torch.bfloat16)

video_predictor.model.to(device=DEVICE)
print(f"[Video] Model dtype  : {next(video_predictor.model.parameters()).dtype}")
print(f"[Video] Model device : {next(video_predictor.model.parameters()).device}")

print(f"\n[Video] Starting session for: {VIDEO_PATH}")
print(f"[Video] offload_video_to_cpu = {OFFLOAD_VIDEO_CPU}")
response = video_predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=VIDEO_PATH,
        offload_video_to_cpu=OFFLOAD_VIDEO_CPU,
    )
)
session_id = response["session_id"]
print(f"[Video] Session started: {session_id}")

print(f"[Video] Adding prompt: '{TEXT_PROMPT_VIDEO}' on frame 0...")
response = video_predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=0,
        text=TEXT_PROMPT_VIDEO,
    )
)
vid_output = response["outputs"]

print("\n" + "=" * 60)
print("VIDEO INFERENCE RESULTS")
print("=" * 60)
if isinstance(vid_output, dict):
    for key, val in vid_output.items():
        if hasattr(val, 'shape'):
            print(f"  '{key}' → shape={val.shape}  dtype={val.dtype}")
        elif isinstance(val, dict):
            print(f"  '{key}' → {val}")
        else:
            print(f"  '{key}' → {type(val).__name__}: {val}")

    # Check masks
    masks_key = next((k for k in ['out_binary_masks', 'masks'] if k in vid_output), None)
    if masks_key is not None:
        masks = vid_output[masks_key]
        if hasattr(masks, 'cpu'):
            masks = masks.cpu().numpy()
        if masks.size == 0:
            print("\n  ⚠  WARNING: Mask array is EMPTY (size=0)!")
        elif masks.sum() == 0:
            print("\n  ⚠  WARNING: Masks exist but are ALL ZEROS!")
        else:
            print(f"\n  ✓  Masks have {masks.sum()} non-zero pixels")

    # Check obj_ids
    if 'out_obj_ids' in vid_output:
        ids = vid_output['out_obj_ids']
        if hasattr(ids, '__len__') and len(ids) == 0:
            print("  ⚠  WARNING: out_obj_ids is EMPTY — model tracked 0 objects!")
        else:
            print(f"  ✓  Tracked object IDs: {ids}")
else:
    print(f"  Output type: {type(vid_output)}")
    print(f"  Output: {vid_output}")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 5 — Visualizer: Image Mask Overlay
# ═══════════════════════════════════════════════════════════════════════
# Requires: Cell 3 (img_masks, img_scores, IMAGE_PATH)

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

orig_img = np.array(Image.open(IMAGE_PATH))
n_cols = min(len(img_masks) + 1, 4)
fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 6))
if not isinstance(axes, np.ndarray):
    axes = [axes]

# Original image
axes[0].imshow(orig_img)
axes[0].set_title('Original', fontsize=13)
axes[0].axis('off')

# Overlay each mask (up to 3)
for i in range(min(len(img_masks), 3)):
    mask = img_masks[i]
    if hasattr(mask, 'cpu'):
        mask = mask.cpu().numpy()
    mask = np.squeeze(mask)

    # Resize mask to match image if needed
    h, w = orig_img.shape[:2]
    if mask.shape != (h, w):
        mask = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)

    overlay = orig_img.copy()
    colored_mask = np.zeros_like(overlay)
    colored_mask[:, :, 0] = 255  # Red channel
    binary = mask > 0
    overlay[binary] = (overlay[binary] * 0.5 + colored_mask[binary] * 0.5).astype(np.uint8)

    nz = int(mask.sum())
    score = img_scores[i].item() if hasattr(img_scores[i], 'item') else img_scores[i]
    axes[i + 1].imshow(overlay)
    axes[i + 1].set_title(f'Mask {i}\nscore={score:.3f}  nz={nz}', fontsize=11)
    axes[i + 1].axis('off')

plt.suptitle(
    f'Image: {IMAGE_PATH} | Prompt: "{TEXT_PROMPT_IMAGE}" | bfloat16={USE_BFLOAT16}',
    fontsize=12
)
plt.tight_layout()
plt.savefig('outputs/diagnostic/image_overlay.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved to outputs/diagnostic/image_overlay.png")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 6 — Visualizer: Video Mask Overlay
# ═══════════════════════════════════════════════════════════════════════
# Requires: Cell 4 (vid_output, VIDEO_PATH)

import cv2
import numpy as np
import os
from IPython.display import HTML, display
import base64

def extract_mask_from_output(output, frame_idx=0):
    """Extract a 2D binary mask for a given frame from the video output dict."""
    mask = None
    if isinstance(output, dict):
        if 'out_binary_masks' in output:
            m = output['out_binary_masks']
            if hasattr(m, 'cpu'):
                m = m.cpu().numpy()
            if m.size > 0:
                m = np.squeeze(m)
                mask = m[0] if m.ndim > 2 else m
    elif isinstance(output, list) and frame_idx < len(output):
        frame_data = output[frame_idx]
        if isinstance(frame_data, dict):
            for key in ['out_binary_masks', 'masks']:
                if key in frame_data:
                    m = frame_data[key]
                    if hasattr(m, 'cpu'):
                        m = m.cpu().numpy()
                    if m.size > 0:
                        m = np.squeeze(m)
                        mask = m[0] if m.ndim > 2 else m
                    break
    return mask

# Read video and write overlay
cap = cv2.VideoCapture(VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"Video: {width}x{height} @ {fps:.1f} fps")

out_path = 'outputs/overlay/video_overlay.mp4'
fourcc   = cv2.VideoWriter_fourcc(*'mp4v')
writer   = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

frame_idx, frames_with_mask = 0, 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    mask = extract_mask_from_output(vid_output, frame_idx)
    if mask is not None:
        if mask.shape != (height, width):
            mask = cv2.resize(mask.astype(np.uint8), (width, height), interpolation=cv2.INTER_NEAREST)
        colored = np.zeros_like(frame)
        colored[:, :, 2] = 255  # Red in BGR
        binary = mask > 0
        if binary.any():
            frames_with_mask += 1
            frame[binary] = cv2.addWeighted(frame, 0.5, colored, 0.5, 0)[binary]

    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

print(f"Processed {frame_idx} frames — {frames_with_mask} had non-empty masks.")
print(f"Saved overlay video to: {out_path}")

# Display inline in Colab
if os.path.exists(out_path):
    with open(out_path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    display(HTML(f'''
    <video width="720" controls>
        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    <p><b>Overlay:</b> {VIDEO_PATH} | Prompt: "{TEXT_PROMPT_VIDEO}" |
       bfloat16={USE_BFLOAT16} | offload={OFFLOAD_VIDEO_CPU}</p>
    '''))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 7 — Pure Mask Export (Image & Video)
# ═══════════════════════════════════════════════════════════════════════
# Saves raw masks for offline inspection.
# Requires: Cell 3 (img_masks) and/or Cell 4 (vid_output)

import os, torch
import numpy as np
from PIL import Image as PILImage
import cv2

os.makedirs('outputs/diagnostic', exist_ok=True)

# ─── Image mask export ────────────────────────────────────────────────
try:
    if len(img_masks) > 0:
        mask = img_masks[0]
        if hasattr(mask, 'cpu'):
            mask = mask.cpu().numpy()
        mask = np.squeeze(mask)
        mask_img = PILImage.fromarray((mask * 255).astype(np.uint8))
        save_path = 'outputs/diagnostic/image_mask.png'
        mask_img.save(save_path)
        print(f"✓ Saved image mask → {save_path}  (non-zero pixels: {int(mask.sum())})")
    else:
        print("⚠ No image masks to export (run Cell 3 first)")
except NameError:
    print("⚠ img_masks not defined — run Cell 3 first")

# ─── Video output export (.pt) ────────────────────────────────────────
try:
    save_path = 'outputs/diagnostic/video_output.pt'
    torch.save(vid_output, save_path)
    print(f"✓ Saved video output → {save_path}")
except NameError:
    print("⚠ vid_output not defined — run Cell 4 first")

# ─── Video mask-only video (white on black) ───────────────────────────
try:
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    mask_vid_path = 'outputs/diagnostic/video_mask_only.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(mask_vid_path, fourcc, fps, (width, height))

    frame_idx, non_empty = 0, 0
    while True:
        ret, _ = cap.read()
        if not ret:
            break

        mask = extract_mask_from_output(vid_output, frame_idx)
        mask_frame = np.zeros((height, width, 3), dtype=np.uint8)
        if mask is not None:
            if mask.shape != (height, width):
                mask = cv2.resize(mask.astype(np.uint8), (width, height), interpolation=cv2.INTER_NEAREST)
            binary = mask > 0
            if binary.any():
                non_empty += 1
            mask_frame[binary] = [255, 255, 255]

        writer.write(mask_frame)
        frame_idx += 1

    cap.release()
    writer.release()
    print(f"✓ Saved mask-only video → {mask_vid_path}  ({non_empty}/{frame_idx} frames with mask)")
except NameError:
    print("⚠ vid_output not defined — run Cell 4 and Cell 6 first")

print("\nAll exports complete. Check outputs/diagnostic/")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 8 — Cleanup (optional)
# ═══════════════════════════════════════════════════════════════════════
# Free GPU memory between test runs.
# After running this, change flags in Cell 2 and re-run from Cell 3/4.

import gc, torch

for var in ['image_model', 'processor', 'video_predictor']:
    if var in globals():
        del globals()[var]
        print(f"  Deleted: {var}")

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    alloc   = torch.cuda.memory_allocated()  / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"\nGPU memory after cleanup:")
    print(f"  Allocated : {alloc:.3f} GiB")
    print(f"  Reserved  : {reserved:.3f} GiB")

print("\nDone. Change flags in Cell 2 and re-run from Cell 3 or Cell 4.")